# Implementing GBrain: A Step-by-Step Tutorial

> Building a self-wiring memory layer for AI agents — the open-source brain Y Combinator's Garry Tan uses to run his own deployments.

**What this notebook covers**

This notebook walks through installing and using [GBrain](https://github.com/garrytan/gbrain) v0.38.2.0 — the markdown-first knowledge layer that powers Garry Tan's OpenClaw and Hermes agents (currently **146,646 pages, 24,585 people, 5,339 companies, and 66 autonomous cron jobs** in his personal deployment).

By the end you'll have:

- A local PGLite brain at `~/.gbrain/brain.pglite` (full Postgres 17 compiled to WASM — zero server, zero Docker)
- A tiny brain repo of markdown notes
- A typed knowledge graph auto-extracted via regex (zero LLM calls)
- A working hybrid-search CLI (vector + BM25 + RRF + ZeroEntropy reranker)
- A 74-tool MCP server wired into Claude Code

All terminal outputs in this notebook were **captured from a live install of GBrain v0.38.2.0** in a clean sandbox. Cells in this notebook use `%%bash` magic — you can re-run any of them on a Linux or macOS machine (Windows users: use WSL2). For the search step you'll need an embedding API key (ZeroEntropy, OpenAI, or Voyage).


## Prerequisites

- macOS or Linux (Windows → WSL2)
- **Bun ≥ 1.3.10** (the JavaScript runtime GBrain ships on; `package.json` declares this minimum)
- An embedding API key from **one** of: ZeroEntropy (default), OpenAI, or Voyage — only needed for Step 6
- Optional: Anthropic API key for multi-query expansion during search

Run this notebook in JupyterLab, VS Code, or Google Colab. Each `%%bash` cell is its own subshell, so the notebook uses absolute paths instead of relying on `cd` carrying across cells.


## Step 1 — Install Bun and GBrain

GBrain runs on Bun. As of v0.38, the canonical install is one global Bun command.

In [ ]:
%%bash
# Install Bun (if you don't already have it)
curl -fsSL https://bun.sh/install | bash
export PATH="$HOME/.bun/bin:$PATH"

# Install GBrain
bun install -g github:garrytan/gbrain

# Verify
bun --version
gbrain --version

1.3.14
gbrain 0.38.2.0


> **If `bun install -g` fails on your system**, the documented fallback is `git clone https://github.com/garrytan/gbrain.git && cd gbrain && bun install && bun link`.

## Step 2 — Initialize your brain

`gbrain init --pglite` provisions a local PGLite database in `~/.gbrain/`. PGLite is full Postgres 17 compiled to WASM — no server, no Docker. We'll pass `--no-embedding` to defer the embedding provider setup until Step 6 (so the rest of the tutorial is reproducible without an API key right away).

In [ ]:
%%bash
export PATH="$HOME/.bun/bin:$PATH"
gbrain init --pglite --no-embedding

Setting up local brain with PGLite (no server needed)...
  --no-embedding: deferred setup — configure with `gbrain config set embedding_model <id>` before import
  Schema version 1 → 85 (81 migration(s) pending)
  [2] slugify_existing_pages...
  [2] ✓ slugify_existing_pages
  [3] unique_chunk_index...
  [3] ✓ unique_chunk_index
  [4] access_tokens_and_mcp_log...
  [4] ✓ access_tokens_and_mcp_log
  ... (81 migrations total) ...
Brain ready at /home/you/.gbrain/brain.pglite
0 pages. Engine: PGLite (local Postgres).


In [ ]:
%%bash
export PATH="$HOME/.bun/bin:$PATH"
gbrain stats

Pages:     0
Chunks:    0
Embedded:  0
Links:     0
Tags:      0
Timeline:  0

By type:


## Step 3 — Create a tiny brain repo

The brain repo is just a directory of markdown files. Each file follows GBrain's **compiled truth + timeline** pattern: a current best-understanding section on top, an append-only evidence trail below.

> **Gotcha I hit while writing this tutorial:** wikilinks must use the **full slug path** (e.g., `[[people/alice-chen]]`, not just `[[alice-chen]]`) for the graph extractor to resolve them. The short form silently produces zero links.

In [ ]:
%%bash
mkdir -p ~/my-brain/people ~/my-brain/companies ~/my-brain/concepts

cat > ~/my-brain/people/alice-chen.md <<'EOF'
---
type: person
title: Alice Chen
tags: [founder, ai-infra]
---

Founder and CEO of [[companies/acme-ai]]. Previously staff engineer at
Google Brain. Focus area: inference optimization for small language models.

---

- 2024-03-12: Met at AI Engineer Summit. Discussed sparse MoE routing.
- 2024-09-04: Announced $12M seed led by Sequoia.
- 2025-01-18: Shipped open-source inference router on GitHub.
EOF

cat > ~/my-brain/companies/acme-ai.md <<'EOF'
---
type: company
title: Acme AI
tags: [startup, inference]
---

YC W24 inference-optimization startup. Founded by [[people/alice-chen]].
Building latency-aware routing for sub-7B models.

---

- 2024-09-04: $12M seed, led by Sequoia.
- 2025-01-18: Open-sourced their inference router.
EOF

cat > ~/my-brain/concepts/inference-optimization.md <<'EOF'
---
type: concept
title: Inference Optimization
tags: [ml-systems]
---

Techniques to reduce latency and cost when serving language models:
quantization, speculative decoding, KV-cache reuse, and request batching.
EOF

ls -R ~/my-brain/

/home/you/my-brain/:
companies
concepts
people

/home/you/my-brain/companies:
acme-ai.md

/home/you/my-brain/concepts:
inference-optimization.md

/home/you/my-brain/people:
alice-chen.md


## Step 4 — Import the repo

`gbrain import` is idempotent — re-running it skips unchanged files (compared by SHA-256 content hash). We pass `--no-embed` so embedding setup stays deferred until Step 6.

In [ ]:
%%bash
export PATH="$HOME/.bun/bin:$PATH"
gbrain import ~/my-brain/ --no-embed

[gbrain phase] import.collect_files start dir=/home/you/my-brain/ strategy=markdown
[gbrain phase] import.collect_files done 2ms files=3
Found 3 markdown files
[import.files] start
[import.files] 3/3 (100%) imported=3 skipped=0 errors=0
[import.files] 3/3 (100%) done

Import complete (0.3s):
  3 pages imported
  0 pages skipped (0 unchanged, 0 errors)
  3 chunks created


In [ ]:
%%bash
export PATH="$HOME/.bun/bin:$PATH"
gbrain list

companies/acme-ai	company	2026-05-22	Acme AI
concepts/inference-optimization	concept	2026-05-22	Inference Optimization
people/alice-chen	person	2026-05-22	Alice Chen


## Step 5 — Wire the knowledge graph

For a first-time import, run the link extractor explicitly to backfill the graph from your wikilinks. This is pure regex + typed inference — **zero LLM calls**.

The inference cascade fires in order: `FOUNDED → INVESTED → ADVISES → WORKS_AT → MENTIONS`. From our three notes it should extract two typed edges:
- `alice-chen --works_at--> acme-ai` (from "Founder and CEO of …")
- `acme-ai --founded--> alice-chen` (from "Founded by …")

In [ ]:
%%bash
export PATH="$HOME/.bun/bin:$PATH"
gbrain extract links --source db

[extract.links_db] start
[extract.links_db] 3/3 (100%)
[extract.links_db] 3/3 (100%) done
Links: created 2 from 3 pages (db source)

Done: 2 links, 0 timeline entries from 3 pages


In [ ]:
%%bash
export PATH="$HOME/.bun/bin:$PATH"
gbrain graph-query people/alice-chen --depth 1

[depth 0] people/alice-chen
  --works_at-> companies/acme-ai (depth 1)


In [ ]:
%%bash
export PATH="$HOME/.bun/bin:$PATH"
gbrain backlinks companies/acme-ai

[
  {
    "from_slug": "people/alice-chen",
    "to_slug": "companies/acme-ai",
    "link_type": "works_at",
    "context": "Founder and CEO of [[companies/acme-ai]]. Previously staff engineer at Google Brain...",
    "link_source": "markdown"
  }
]


This is the difference between vector search and structured retrieval. *"Who works at Acme AI?"* is now a one-hop typed-edge traversal, not a similarity score. That structural channel is what produces GBrain's **+31.4-point P@5 lead** over the graph-disabled variant on BrainBench (the project's own 240-page benchmark corpus).

## Step 6 — Search

GBrain ships two search verbs:

1. **`gbrain search`** — keyword-only (BM25 on Postgres `tsvector`). **Works without embeddings.**
2. **`gbrain query`** — full hybrid: vector (HNSW on pgvector) + BM25 + Reciprocal Rank Fusion + optional Anthropic Haiku multi-query expansion + ZeroEntropy reranker. **Needs embeddings.**

Three search modes ship out of the box — `conservative`, `balanced`, `tokenmax` — bundling the cost/quality knobs into one config key. Default: `balanced` with reranker on. The RRF formula is `score = sum(1 / (60 + rank))`.

In [ ]:
%%bash
export PATH="$HOME/.bun/bin:$PATH"
gbrain search "inference" 

[0.3648] companies/acme-ai -- YC W24 inference-optimization startup. Founded by [[people/alice-chen]]...
[0.3648] people/alice-chen -- Founder and CEO of [[companies/acme-ai]]. Previously staff engineer at Google Brain...


For the full hybrid `query` pipeline, configure an embedding provider and backfill embeddings. **This requires an API key.** Pick one of the three supported providers:

In [ ]:
%%bash
# Pick ONE — set the matching API key in your shell first, e.g.:
#   export OPENAI_API_KEY=sk-...
#   export ZEROENTROPY_API_KEY=ze-...
#   export VOYAGE_API_KEY=pa-...

export PATH="$HOME/.bun/bin:$PATH"
gbrain config set embedding_model openai:text-embedding-3-large
gbrain embed --all
gbrain query "who works on small-model inference?" 

## Step 7 — Connect to Claude Code via MCP

GBrain exposes **74 tools** over the Model Context Protocol via stdio (I counted with `gbrain --tools-json | jq length` — the README's "30+" is understated). The canonical setup is one command, not a hand-edited JSON file:

In [ ]:
%%bash
export PATH="$HOME/.bun/bin:$PATH"

# Claude Code: register gbrain as a local stdio MCP server
claude mcp add gbrain -- gbrain serve

# Verify
claude mcp list

Added stdio MCP server gbrain with command: gbrain serve to local config
gbrain  stdio  gbrain serve


**Cursor and Windsurf** use the standard MCP JSON config in their respective settings. The server spec is identical:

```json
{
  "mcpServers": {
    "gbrain": { "command": "gbrain", "args": ["serve"] }
  }
}
```

**Claude Desktop** uses `claude_desktop_config.json` for *local* stdio servers with the same spec. *Remote* HTTP MCP servers must be added through Settings → Integrations with a bearer token.

The 74 tools are plain snake_case: `get_page`, `put_page`, `delete_page`, `list_pages`, `search`, `query`, `add_link`, `get_backlinks`, `add_tag`, `add_timeline_entry`, and 64 more. Once the MCP server is wired in, ask Claude Code something like *"search my brain for inference optimization"* and it'll route through the `search` tool.

For remote access from any machine, swap stdio for HTTP:

```bash
gbrain serve --http --port 8787
# Bearer auth, default-deny CORS, two-bucket rate limit, per-request audit log.
# Postgres-only by design (PGLite is local-only).
```

## Step 8 — Let the brain run itself

GBrain v0.36.4 added an autopilot loop. One command computes a dependency-ordered remediation plan, submits each step as a Minion job, re-checks the brain's health score between steps, and refuses to spend past your cost cap:

```bash
gbrain doctor --remediate --yes --target-score 90 --max-usd 5
```

Or run it continuously:

```bash
gbrain autopilot --install        # cron-driven, 5-minute tick
```

For ad-hoc background work, the **Minions** queue takes shell jobs and LLM subagent jobs side by side:

```bash
gbrain jobs submit sync --params '{}' --follow
gbrain jobs stats
```

> **PGLite caveat:** `gbrain jobs supervisor` (the auto-restarting worker daemon) is **Postgres-only**. PGLite's exclusive file lock blocks the separate worker process — the CLI rejects with a clear error if `config.engine === 'pglite'`. If you're on PGLite, use inline `--follow` jobs, or run `gbrain migrate --to supabase` first.

Routing rule: deterministic work (pull tweets, parse JSON, write a page) goes to Minions; judgment work (triage an inbox, assess priority) goes to LLM sub-agents.

## Where to go next

- **One-line capture** (new in v0.38): `gbrain capture "the thought I want to remember"` lands in `inbox/YYYY-MM-DD-<hash>`. Also accepts `--file`, `--stdin`, and webhook ingestion via `gbrain serve --http /ingest`.
- **Migrate to Supabase** when your brain outgrows local (PGLite is good up to ~50K pages): `gbrain migrate --to supabase`.
- **Ingest real data** with recipes for voice (Twilio + OpenAI Realtime), email + calendar webhooks, and 16 embedding providers.
- **Run the benchmarks** in the sibling repo [gbrain-evals](https://github.com/garrytan/gbrain-evals): BrainBench (synthetic) and `gbrain eval longmemeval` (public).
- **Author your own skills.** A skill is a fat markdown file that encodes a workflow — triggers, checks, quality gate. `gbrain check-resolvable` validates the skill tree for reachability / MECE / DRY.

The deeper bet behind GBrain is that **thin harness, fat skills** beats thin skills behind a fat agent. The runtime stays small; the intelligence lives in markdown files the agent reads at decision time. Each commit you make to your brain repo is permanent context your agent inherits the next time it wakes up. The longer you run it, the smarter it gets.

---

*Tutorial built against GBrain v0.38.2.0 (commit `3de06b6c`). Repository: [github.com/garrytan/gbrain](https://github.com/garrytan/gbrain). License: MIT.*
